# Лекция 9 — Работа с базами данных в Python (модуль `sqlite3`): задачи для самостоятельной работы

**Темы:** подключение к SQLite и курсор, `execute()` / `executemany()` / `executescript()`, `fetchone()` / `fetchmany()` / `fetchall()` и итерация по курсору, CRUD (`CREATE` / `INSERT` / `SELECT` / `UPDATE` / `DELETE`), типы данных, параметризованные запросы против SQL-инъекций, транзакции (`commit` / `rollback`, `with connection`), ограничения (`PRIMARY KEY`, `UNIQUE`, внешние ключи), `LIKE` / `BETWEEN` / `IN`, агрегаты, `GROUP BY` / `HAVING`, `JOIN`, `LIMIT`/`OFFSET`, pandas и `to_sql` / `read_sql_query`.

- **Задачи 1–10** — простые
- **Задачи 11–20** — средние
- **Задачи 21–30** — сложные

Все решения должны работать **без интернета** и без сторонних библиотек, кроме `pandas` и `numpy` (только в задачах 24 и 30): достаточно стандартного модуля `sqlite3`.

Файловые базы данных создавайте в `tempfile.TemporaryDirectory()` (или `tempfile.mkstemp()`) и удаляйте в конце ячейки; там, где это возможно, достаточно базы в памяти — `sqlite3.connect(":memory:")`. В каждой задаче цель проверяется `assert`-ом.


## Простые задачи


### 1. Подключение к базе и курсор
Подключитесь к базе в памяти (`:memory:`), получите курсор и выполните запрос `SELECT sqlite_version()`. Проверьте, что курсор имеет тип `sqlite3.Cursor`, а версия — непустая строка, начинающаяся с `'3.'`.

Затем закройте соединение и убедитесь, что любое обращение к «осиротевшему» курсору вызывает `sqlite3.ProgrammingError`.

```python
import sqlite3

con = sqlite3.connect(":memory:")
cur = con.cursor()
# 1) execute + fetchone -> версия
# 2) con.close(), потом попытка cur.execute("SELECT 1") в try/except
```

**Подсказка:** `execute()` возвращает курсор с результатом, `fetchone()` отдаёт первую строку-кортеж, поэтому версия — это `cur.fetchone()[0]`.


### 2. CREATE TABLE: схема таблицы
Создайте таблицу `students` со столбцами `id INTEGER PRIMARY KEY`, `name TEXT NOT NULL`, `score INTEGER`. Проверьте результат двумя способами: имя таблицы есть среди имён из `sqlite_master` (при `type = 'table'`), а список столбцов из `PRAGMA table_info(students)` равен `['id', 'name', 'score']`.

Повторный `CREATE TABLE students` должен вызвать `sqlite3.OperationalError`, а `CREATE TABLE IF NOT EXISTS students` — пройти молча.

**Подсказка:** `sqlite_master` — служебная таблица самой SQLite со списком объектов схемы; `PRAGMA table_info(имя)` возвращает по строке на столбец, имя столбца — на позиции 1.


### 3. INSERT и commit: данные переживают закрытие соединения
В файловой БД (файл внутри `tempfile.TemporaryDirectory()`) создайте таблицу `students (id INTEGER PRIMARY KEY, name TEXT, score INTEGER)`, вставьте одну запись `('Иванов', 85)`, сделайте `commit()` и закройте соединение.

Затем откройте этот же файл **вторым** соединением и убедитесь, что в таблице ровно одна строка и она равна `(1, 'Иванов', 85)`.

**Подсказка:** без `commit()` изменения не считаются зафиксированными и другому соединению не видны; `TemporaryDirectory` удалит файл БД автоматически при выходе из `with`.


### 4. fetchone, fetchmany, fetchall
Создайте таблицу из трёх столбцов и заполните её пятью строками (`score` = 1..5). Выполните **один** `SELECT * FROM ... ORDER BY id` и заберите результат тремя методами: сначала `fetchone()`, затем `fetchmany(2)`, затем `fetchall()`.

Проверьте, что первая строка — `(1, ..., 1)`, второй вызов вернул ровно 2 строки, третий — 2 оставшиеся, а следующий `fetchone()` возвращает `None` (курсор исчерпан).

**Подсказка:** все `fetch*` методы сдвигают «читалку» курсора вперёд и не повторяют строки; повторный `execute()` того же `SELECT` возвращает результат с самого начала.


### 5. Параметризованный INSERT: rowcount и lastrowid
Создайте таблицу `users (id INTEGER PRIMARY KEY AUTOINCREMENT, login TEXT UNIQUE, full_name TEXT)`. Вставьте две записи через плейсхолдеры `?` и проверьте после каждой вставки, что `cur.rowcount == 1`, а `cur.lastrowid` — это номер вставленной строки.

Отдельно вставьте ФИО, содержащее апостроф и запятую (`"O'Brien, Patrick"`), и прочитайте его обратно параметризованным запросом: строка должна вернуться ровно такой, какой вы её передали.

**Подсказка:** значение передаётся вторым аргументом `execute(sql, (значение,))` и никогда не «вклеивается» в текст SQL — поэтому апострофы внутри данных безвредны.


### 6. UPDATE: с WHERE и без него
Заполните таблицу `grades (id INTEGER PRIMARY KEY, name TEXT, score INTEGER)` тремя строками со значениями `score` 60, 70 и 80.

Проверьте три случая: `UPDATE ... WHERE name = ?` меняет ровно одну строку (`cur.rowcount == 1`); `UPDATE` **без** `WHERE` затрагивает все строки (`cur.rowcount == 3`); условие, которое ничего не находит, даёт `cur.rowcount == 0`. После двух первых обновлений значения `score` по порядку id должны быть `[65, 100, 85]`.

**Подсказка:** `cur.rowcount` после `UPDATE`/`DELETE` — число реально изменённых (удалённых) строк; именно так заметают «забытый WHERE».


### 7. DELETE: с WHERE и без него
На той же таблице `grades` удалите строки с `score < 80` параметризованным запросом и проверьте `cur.rowcount` и остаток. Затем удалите **все** строки командой `DELETE FROM grades` без `WHERE` и убедитесь, что `SELECT COUNT(*)` вернул 0.

Отдельно проверьте, что `DELETE` с условием, которое ничего не находит, не ошибка, а `rowcount == 0`.

**Подсказка:** `DELETE FROM t` без `WHERE` — это «очистить таблицу», а не ошибка; поэтому удалённые данные стоит предварительно посчитать тем же `SELECT` с тем же `WHERE`.


### 8. executescript: схема и справочник одним куском
Одним вызовом `executescript()` создайте две таблицы (`cities`, `shops`) и заполните их начальными данными несколькими `INSERT`. Проверьте, что `executescript()` вернул объект `sqlite3.Cursor`, в `cities` ровно 3 строки, в `shops` ровно 4 строки, а обе таблицы появились в `sqlite_master`.

**Подсказка:** `executescript()` принимает текст с несколькими SQL-командами, разделёнными `;`, и перед выполнением делает неявный `COMMIT` незакрытой транзакции; для одной команды с параметрами он не подходит — там нужен `execute()`.


### 9. SELECT с ORDER BY и LIMIT
Заполните таблицу оценками пяти студентов: `('Анна', 'Москва', 88)`, `('Борис', 'Томск', 72)`, `('Вера', 'Тула', 95)`, `('Глеб', 'Томск', 88)`, `('Дина', 'Москва', 72)`.

Выберите `name, score` с сортировкой `score DESC`, а при равных баллах — по имени. Ожидаемый порядок: `['Вера', 'Анна', 'Глеб', 'Борис', 'Дина']`. Проверьте это, проверьте, что балли идут невозрастающе, и что `LIMIT 3` даёт тройку лидеров.

**Подсказка:** `ORDER BY score DESC, name` — два ключа сортировки; `LIMIT n [OFFSET m]` отсекает лишние строки уже на стороне SQLite.


### 10. Типы данных: SQLite ↔ Python
Создайте таблицу `t (i INTEGER, r REAL, s TEXT, b BLOB)`. Проверьте «круговой проход» типов: переданные `42`, `3.5`, `'привет'`, `b'\x00\x01'` должны вернуться как `int`, `float`, `str`, `bytes`. Пропущенные значения должны вернуться как `None`.

Отдельно вставьте строку `'123'` в `INTEGER`-столбец и убедитесь, что SQLite по аффинитету типа вернул **целое** `123`, а не строку.

**Подсказка:** в SQLite пять хранимых классов (NULL, INTEGER, REAL, TEXT, BLOB) и динамическая типизация: аффинитет столбца пытается привести вставленное значение к «своему» типу, поэтому `True` в таблице окажется единичкой, а `'123'` в `INTEGER` — числом.


## Средние задачи


### 11. executemany вместо цикла execute
Создайте две одинаковые таблицы `a` и `b`. Загрузите 2000 записей `(x, y)` дважды: в таблицу `a` — одним `executemany()` со списком кортежей, в таблицу `b` — циклом из 2000 вызовов `execute()`. Замерьте оба варианта через `time.perf_counter()`.

Проверьте, что в обеих таблицах по 2000 строк и что содержимое совпадает (`ORDER BY id`), а `executemany()` не медленнее цикла.

**Подсказка:** `executemany(sql, list_of_tuples)` компилирует запрос один раз и подставляет параметры для каждой строки; количество плейсхолдеров `?` должно совпадать с длиной кортежей.


### 12. with connection: менеджер транзакции
В файловой БД создайте таблицу `accounts (id INTEGER PRIMARY KEY, login TEXT, amount INTEGER)`. Выполните вставку внутри блока `with con:` **без** явного `commit()` и проверьте, что сразу после выхода из блока запись уже видна из **второго** соединения (то есть транзакция закоммичена).

Отдельно убедитесь, что `with con:` соединение **не** закрывает: после блока обычный запрос к `con` работает, и только `con.close()` его завершает.

**Подсказка:** менеджер контекста у соединения отвечает за транзакцию (`commit` при успехе, `rollback` при исключении), но не за ресурс — `close()` (или `contextlib.closing`) остаётся ручным.


### 13. rollback: откат после ошибки
Зафиксируйте (`commit`) три строки в таблице `events`. Затем в блоке `try/except` выполните вставку новой строки и намеренно поднимите `RuntimeError` до выхода из `with con:` — транзакция должна откатиться.

Проверьте откатившееся состояние: из того же соединения в таблице ровно 3 строки `['start', 'work', 'stop']`, и из **второго** соединения `COUNT(*)` тоже равен 3 (откаченные данные не попали в файл).

Повторите то же самое явным `con.rollback()` без `with` и снова получите 3 строки.

**Подсказка:** выход из `with con:` по исключению сам вызывает `rollback()`; проверить откат нужно обязательно «свежим» запросом, а не значением переменной в памяти.


### 14. LIKE: поиск по подстроке
Заполните таблицу `people (name TEXT)` данными: `Иванов`, `Иваненко`, `Петров`, `Сидоров`, `иванова`, `Ivanov`, `ivanov`.

Параметризованными запросами с `LIKE ?` (шаблон собирайте в Python: `'Иван%'`, `'%ов'`, `'NOT LIKE'`) найдите: всех на `Иван` — ровно 2 строки; всех с окончанием `ов` — ровно 3 строки; всех, кто **не** заканчивается на `ов` — ровно 4 строки. Отдельно покажите, что `LIKE 'ivan%'` находит обе латинские фамилии, а `LIKE 'иван%'` — только одну кириллическую, то есть для латиницы `LIKE` регистронезависим, а для кириллицы — нет.

**Подсказка:** `%` — любое число символов, `_` — ровно один; шаблон — это **параметр**, а не часть текста SQL; регистронезависимость `LIKE` в SQLite по умолчанию действует только для ASCII.


### 15. BETWEEN и IN со списком значений
Заполните таблицу `orders (id INTEGER PRIMARY KEY, client TEXT, amount INTEGER, status TEXT)` восемью строками на своё усмотрение (со статусами из набора `новый`, `оплачен`, `доставлен`, `отменён`).

Проверьте: `WHERE amount BETWEEN ? AND ?` включает границы (`amount` ровно на нижней границе попадает в выборку); для фильтра по нескольким статусам соберите `IN` из нужного числа плейсхолдеров (`",".join("?" * len(statuses))`) и сверьте `COUNT(*)` с подсчётом на Python-стороне.

**Подсказка:** `BETWEEN` симпатичен тем, что в текст запроса не надо вклеивать числа; для `IN` число `?` обязано равняться числу передаваемых значений, поэтому список плейсхолдеров строят динамически.


### 16. PRIMARY KEY и UNIQUE: ловим IntegrityError
Создайте таблицу `clients (id INTEGER PRIMARY KEY, email TEXT UNIQUE NOT NULL, name TEXT)` и вставьте одну запись. Проверьте, что: повторный вставка того же `email` поднимает `sqlite3.IntegrityError`; после `con.rollback()` число строк не изменилось; вставка `NULL` в `email` тоже поднимает `sqlite3.IntegrityError`; а `INSERT OR IGNORE` с дубликатом **не** бросает исключения и не добавляет строк.

**Подсказка:** ловите `sqlite3.IntegrityError` (потомок `sqlite3.DatabaseError`), после ошибки обязательно `rollback()` — транзакция в непонятном состоянии; текст ошибки содержит `UNIQUE constraint failed: таблица.столбец`.


### 17. Агрегаты и GROUP BY из Python
Заполните таблицу `emp (name TEXT, city TEXT, salary INTEGER)` восемью строками так, чтобы в одном городе было 3 человека, в другом 2, и ещё два города по 1 человеку.

Соберите в словарь `stats` результат запроса `SELECT city, COUNT(*), SUM(salary), ROUND(AVG(salary), 2) FROM emp GROUP BY city`; отдельно запросите города с ФОТ больше порога через `HAVING SUM(salary) > ?`. Проверьте конкретные значения на своих данных и сверьте `COUNT(*)` с `len(emp)`.

**Подсказка:** `HAVING` фильтрует уже сгруппированные строки (фильтр по агрегату), а `WHERE` — исходные; в `GROUP BY` можно указать несколько столбцов, и тогда ключом словаря станет кортеж.


### 18. Динамические фильтры: собираем WHERE из частей
Реализуйте `search_users(con, name=None, city=None, min_score=None, sort_by="id", limit=None)`, который собирает `SELECT` из **непустых** фильтров: список условий соединяется через `AND`, значения уходят вторым аргументом `execute` списком параметров, сортировка берётся из белого списка `{"id", "name", "score"}`, а `limit` добавляет `LIMIT ?`.

Проверьте, что пустой вызов возвращает всех, комбинация фильтров возвращает ожидаемое подмножество, `sort_by="salary"` поднимает `ValueError`, и что имя с апострофом (`"O'Brien"`) работает как параметр, не ломая запрос.

**Подсказка:** собирайте два параллельных списка — `where_parts` и `params`, а затем `sql = "SELECT * FROM users" + (" WHERE " + " AND ".join(where_parts) if where_parts else "")`; имена столбцов параметрами передать нельзя, поэтому для `sort_by` нужен белый список.


### 19. Курсор как итератор: много строк без раздувания памяти
Заполните таблицу `logs (id INTEGER PRIMARY KEY, level TEXT, msg TEXT)` пятью тысячами строк, где `level` циклически принимает `INFO`, `WARN`, `ERROR`.

Пройдите по курсору `SELECT level, msg FROM logs` простым `for level, msg in cur:` (без `fetchall()`) и посчитайте общее число строк и число `ERROR`. Затем прочитайте `SELECT id FROM logs` пачками через `fetchmany()` с `cur.arraysize = 1000` и проверьте, что пачек было ровно 5, а `cur.description` даёт имена столбцов.

**Подсказка:** курсор — итерируемый объект, строки приходят по одной; `fetchmany()` без аргумента использует `cur.arraysize`, и цикл `while True` с выходом по пустому чанку — стандартный способ читать большими порциями.


### 20. row_factory = sqlite3.Row: доступ по имени столбца
Заполните небольшую таблицу и включите `con.row_factory = sqlite3.Row`. Прочитав строку, получите значения **по имени** (`row["name"]`), **по индексу** (`row[1]`), посмотрите `row.keys()` и превратите строку в словарь через `dict(row)`.

Затем задайте собственный `row_factory` — лямбду, которая возвращает `dict`, — и убедитесь, что `fetchall()` отдаёт список словарей, готовый для `json.dumps` или `DataFrame`.

**Подсказка:** в лямбде-фабрике имена столбцов берутся из `cur.description` (`[d[0] for d in cur.description]`) и склеиваются со строкой-кортежем через `zip`; `sqlite3.Row` при этом поддерживает и `row["col"]`, и `row[i]`.


## Сложные задачи


### 21. SQL-инъекция: ломаем запрос f-строкой и защищаемся `?`
В **отдельной временной** базе создайте таблицу `users (login TEXT, name TEXT, role TEXT)` с тремя строками: два пользователя с ролью `user` и один `admin`.

Покажите три вещи: (1) запрос, собранный f-строкой от «пользовательского ввода» `"admin' --"`, выдаёт строку администратора, хотя условие `AND role = 'user'` должно было её отсечь (символы `--` комментируют остаток запроса); (2) тот же ввод как **параметр** (`?`) не находит ничего; (3) строка-«вредонос» вроде `"x'; DELETE FROM users; --"`, переданная параметром, остаётся обычным текстом — таблица и все 3 строки целы.

Проверьте всё `assert`-ами, в конце закройте соединение. Никаких реальных удалений и системных команд: база временная, ущерб исключительно учебный.

**Подсказка:** `--` в SQL начинает комментарий, поэтому в уязвимом запросе «дописанная» часть условия просто перестаёт существовать; `execute()` к тому же разрешает только один оператор за раз, что не даёт выполнить `DELETE` из строки-параметра.


### 22. Реляционная схема: внешние ключи и JOIN из Python
Создайте две таблицы: `authors (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE)` и `books (id INTEGER PRIMARY KEY, author_id INTEGER NOT NULL REFERENCES authors(id) ON DELETE CASCADE, title TEXT, pages INTEGER)`.

Проверьте: SQLite по умолчанию **не** проверяет внешние ключи, и только после `PRAGMA foreign_keys = ON` вставка книги с несуществующим `author_id` поднимает `sqlite3.IntegrityError` (не забудьте, что `PRAGMA` надо выполнить до начала транзакции). Затем сделайте `DELETE` автора и убедитесь, что его книги удалились каскадно. Наконец, выберите из Python список пар `(имя автора, заголовок)` через `JOIN ... ON` с сортировкой.

**Подсказка:** `PRAGMA foreign_keys` — соединение-уровня настройка, её нужно включать заново для каждого `connect()`; проверить «без pragma» удобно вторым соединением к той же файловой БД.


### 23. Транзакция «перевод денег»: инвариант при ошибке
Создайте таблицу `accounts (login TEXT PRIMARY KEY, amount INTEGER NOT NULL)` с двумя счетами по 500 (итого 1000). Реализуйте `transfer(con, src, dst, summa)`: списать, проверить, что баланс не ушёл в минус, зачислить, `commit()`; при любой ошибке — `rollback()` и вернуть `False`.

Проверьте успешный перевод (балансы изменились верно, сумма осталась 1000) и неудачный перевод больше баланса: функция вернула `False`, оба баланса **в точности** как до попытки, сумма инвариантна, а из второго соединения в файле БД тоже видно старое состояние.

**Подсказка:** именно атомарность транзакции даёт «либо оба UPDATE, либо ни одного»; промежуточное состояние (деньги списаны, но не зачислены) не должно быть видно наружу даже на долю секунды после отката.


### 24. CSV → pandas → SQLite и обратно
В `TemporaryDirectory()` запишите таблицу продаж (8 строк, столбцы `date, region, product, qty, price`) модулем `csv`, прочитайте её `pd.read_csv` и выгрузите `DataFrame.to_sql("sales", con, index=False, if_exists="replace")` в SQLite-соединение.

Проверьте, что `SELECT COUNT(*)` из таблицы равен числу строк датафрейма; прочитайте обратно `pd.read_sql_query` с фильтром по региону и `params`; сверьте сумму `qty * price` из SQL-агрегата с pandas-аналогом; повторный `to_sql(..., if_exists="replace")` не должен плодить строки.

**Подсказка:** для SQLite `to_sql`/`read_sql_query` работают с обычным `sqlite3.Connection` (SQLAlchemy не нужна); `read_sql_query(..., params=(...,))` умеет принимать `?`-плейсхолдеры.


### 25. Мини-DAO: класс-обёртка над sqlite3 и CRUD-тесты
Реализуйте класс `Repo` (таблица `users (id INTEGER PRIMARY KEY, login TEXT NOT NULL UNIQUE, name TEXT, score INTEGER NOT NULL DEFAULT 0)`) с методами: `create(login, name, score=0) -> int`, `get(user_id) -> dict | None`, `all() -> list[dict]`, `update_score(user_id, score) -> int`, `delete(user_id) -> int`, `top(limit=3) -> list[dict]`, `close()`. Внутри — `row_factory = sqlite3.Row` и возврат словарей.

Прогоните CRUD-тесты: `create` возвращает id 1, 2, 3; дубликат `login` поднимает `sqlite3.IntegrityError`; `get` несуществующего id — `None`; `update_score` возвращает `rowcount`; `delete` уменьшает `len(all())`; `top(2)` отсортирован по убыванию баллов (при равных — по логину).

**Подсказка:** пусть каждый публичный метод сам делает `commit()` после записи — так DAO удобно использовать из notebook; `dict(row)` превращает `sqlite3.Row` в словарь, а `None` от `fetchone()` означает «нет записи».


### 26. Кэш запроса: меньше обращений к базе
Реализуйте класс `CachedReader(con)` с методами `fetch(sql, params=())`, `execute_write(sql, params=())` и счётчиком `db_calls`. Чтение с теми же `sql` и параметрами должно второй раз обслуживаться из словаря-кэша без обращения к базе, а любая запись — очищать кэш целиком и делать `commit()`.

Проверьте: три одинаковых чтения подряд дают `db_calls == 1`; после записи `db_calls == 2`, а результат чтения изменился; чтение с другими параметрами считается отдельным ключом кэша; результат в кэше нельзя «испортить» — возвращайте копию списка.

**Подсказка:** ключ кэша — кортеж `(sql, tuple(params))`; если вернуть наружу тот же список, что лежит в кэше, вызывающий код может его изменить и отравить кэш, поэтому `return list(rows)`.


### 27. ROWID и пагинация LIMIT/OFFSET
Создайте таблицу `notes (text TEXT)` **без** `INTEGER PRIMARY KEY` и вставьте 25 строк `заметка-01 ... заметка-25`. Убедитесь, что скрытый `rowid` существует и доступен в `SELECT rowid, text FROM notes`.

Пройдите таблицу постранично (`page_size = 4`) запросом `SELECT rowid, text FROM notes ORDER BY rowid LIMIT ? OFFSET ?`, соберите все страницы и проверьте: страница ровно 7, всего собрано 25 строк, все `rowid` уникальны, порядок совпадает с ожидаемым, третья страница равна срезу `[8:12]` полного списка.

**Подсказка:** у любой таблицы без `WITHOUT ROWID` есть скрытый `rowid` — стабильный «якорь» для сортировки; без `ORDER BY` SQLite не гарантирует порядок, и при пагинации одни строки могут повториться, а другие потеряться.


### 28. Загрузка с обновлением: INSERT OR IGNORE / OR REPLACE / ON CONFLICT
В файловой БД создайте `price (code TEXT PRIMARY KEY, title TEXT NOT NULL, price REAL NOT NULL)`. Реализуйте `load_prices(con, csv_path, mode)`, которая читает CSV модулем `csv.DictReader` и загружает его в таблицу одним `executemany` в режиме `plain` (обычный `INSERT`), `ignore` (`INSERT OR IGNORE`), `replace` (`INSERT OR REPLACE`) или `upsert` (`INSERT ... ON CONFLICT(code) DO UPDATE SET title = excluded.title, price = excluded.price`).

Проверьте: первый файл загружает 4 строки; обычный `INSERT` повторного файла поднимает `sqlite3.IntegrityError`, и после `rollback()` в таблице те же 4 строки; в режиме `ignore` цены существующих кодов **не** обновились (новые коды при этом добавились); в режимах `replace` и `upsert` обновились, а всего строк стало 6 (появились два новых кода).

**Подсказка:** `excluded` — это «вставляемая» строка в контексте конфликта; `INSERT OR REPLACE` удаляет старую строку и вставляет новую (сбрасывая значения по умолчанию других столбцов), а `ON CONFLICT DO UPDATE` меняет только перечисленные столбцы.


### 29. Резервная копия и восстановление: iterdump и backup
Создайте файловую БД с двумя связанными таблицами и данными. Сделайте текстовый дамп всей базы через `"\n".join(con.iterdump())`, сохраните его в `.sql`-файл и восстановите в **новую** пустую БД одним `executescript()`.

Проверьте, что после восстановления совпали имена таблиц, содержимое обеих таблиц и `PRAGMA integrity_check` вернул `'ok'`. Отдельно сделайте «горячее» копирование вторым способом — `con.backup(target_con)` — и сверьте `COUNT(*)`.

**Подсказка:** `iterdump()` отдаёт SQL-текст, которым базу можно пересоздать (портативно ичитаемо человеком), а `backup()` копирует БД на лету через API SQLite; перед дампом сделайте `commit()`, иначе незакрытая транзакция не попадёт в дамп.


### 30. Отчёт по-взрослому: JOIN + GROUP BY + HAVING и сверка с pandas
Соберите схему `managers (id, name)`, `regions (id, name)`, `orders (id, manager_id, region_id, amount, status)` и заполните её данными (10–12 заказов, статусы `open`/`closed`, один заказ у «осиротевшего» менеджера, чтобы проверить `JOIN`).

SQL-запросом с `JOIN` двух связей, `WHERE status = ?`, `GROUP BY` по региону и `HAVING COUNT(*) >= ?` получите отчёт `(регион, имя менеджера, число заказов, сумма, средний чек)`; результат запишите через `executemany` в таблицу `report` и прочитайте обратно. Сверьте SQL-отчёт с pandas-аналогом (`pd.read_sql_query` всей таблицы, `groupby`, `agg`) и убедитесь, что суммы и количества совпадают.

**Подсказка:** фильтруйте статус до `GROUP BY` (в `WHERE`), а агрегаты — в `HAVING`; в pandas считайте агрегаты по той же «сырой» выборке и сравнивайте через `round(..., 2)`, потому что `AVG` в SQL и `mean()` в pandas могут разойтись в последнем бите.


## Где потренироваться дальше

- [Документация модуля `sqlite3`](https://docs.python.org/3/library/sqlite3.html) — официальный справочник: соединения, курсоры, транзакции, `row_factory`
- [Язык запросов SQLite](https://www.sqlite.org/lang.html) — синтаксис SQL-диалекта SQLite (`SELECT`, `INSERT`, `UPDATE`, `DELETE`)
- [Внешние ключи в SQLite](https://www.sqlite.org/foreignkeys.html) — почему `PRAGMA foreign_keys = ON` нужно включать вручную
- [`DataFrame.to_sql`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html) и [`read_sql_query`](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html) — pandas и реляционные базы
- [Тренажёр SQL на русском](https://www.sql-ex.ru/) — задачи на `SELECT`, `JOIN`, группировки и подзапросы
- [SQLBolt](https://sqlbolt.com/) — короткие интерактивные уроки по SQL
- [SQL-инъекция (Википедия)](https://ru.wikipedia.org/wiki/SQL-%D0%B8%D0%BD%D1%8A%D0%B5%D0%BA%D1%86%D0%B8%D1%8F) — виды инъекций и защитные приёмы
- [Браузер БД DB Browser for SQLite](https://sqlitebrowser.org/) — посмотреть глазами на файл базы, созданный из Python
